In [3]:
import numpy as np
import gc
import tensorflow as tf
from tensorflow import keras
from keras import models,layers
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


In [4]:
dataset = np.load("//kaggle/input/datasets/hassanrajput189/nails-training-dataset/nails_disease_dataset_training_multi_class.npz")
X_train_nails = dataset["X_train"]
X_train_nails.shape

(4908, 224, 224, 3)

In [5]:
dataset = np.load("//kaggle/input/datasets/hassanrajput189/nails-testing-dataset/nails_disease_dataset_testing_multi_class.npz")
X_test_nails = dataset["X_test"]
X_test_nails.shape

(392, 224, 224, 3)

In [6]:
dataset = np.load("/kaggle/input/datasets/hassanrajput189/no-nails-dataset-training/no_knee_images.npz")
X_train_no_nails = dataset["X"]
X_train_no_nails.shape

(5868, 224, 224, 3)

In [7]:
dataset = np.load("/kaggle/input/datasets/hassanrajput189/no-nails-dataset-testing/no_knee_images_test.npz")
X_test_no_nails = dataset["X"]
X_test_no_nails.shape

(1357, 224, 224, 3)

In [8]:
y_train_nails = np.full(X_train_nails.shape[0],1)
y_train_no_nails = np.full(X_train_no_nails.shape[0],0)
y_test_nails = np.full(X_test_nails.shape[0],1)
y_test_no_nails = np.full(X_test_no_nails.shape[0],0)
y_train_nails.shape,y_train_no_nails.shape,y_test_nails.shape,y_test_no_nails.shape

((4908,), (5868,), (392,), (1357,))

In [9]:
X_train_combined = np.concatenate([X_train_no_nails, X_train_nails])
X_test_combined = np.concatenate([X_test_no_nails, X_test_nails])
y_train_combined = np.concatenate([y_train_no_nails, y_train_nails])
y_test_combined = np.concatenate([y_test_no_nails, y_test_nails])

X_train_combined.shape, y_train_combined.shape, X_test_combined.shape, y_test_combined.shape

((10776, 224, 224, 3), (10776,), (1749, 224, 224, 3), (1749,))

In [ ]:
np.savez_compressed("nails_disease_gatekeeper_training_dataset",X_train = X_train_combined,y_train = y_train_combined)

In [ ]:
np.savez_compressed("nails_disease_gatekeeper_testing_dataset",X_test = X_test_combined,y_test = y_test_combined)

In [10]:
num_classes = 1
base_model = MobileNetV3Small(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224, 3),
)

I0000 00:00:1792602092.051251      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792602092.054762      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [12]:
base_model.trainable = False

In [18]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu",
    ),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="sigmoid")
])


In [13]:
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_combined), y=y_train_combined)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)

reduce_lr = ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-10, verbose=1
)
early_stop = EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True,verbose=1
)

Class weights: {0: np.float64(0.918200408997955), 1: np.float64(1.097799511002445)}


In [14]:
def train_evaluate_model(
    optimizer,
    loss_function,
    X_train,
    y_train,    
    X_test,
    y_test,
    epochs,
    class_weights, 
    callbacks   
    ):
    model.compile(
    optimizer=optimizer,
    loss=loss_function,
    metrics=["accuracy"]
    )
    history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    validation_data=(X_test, y_test),
    class_weight=class_weights,
    callbacks=callbacks,
    )
    
    y_pred = model.predict(X_test)
    y_pred = model.predict(X_test)

    # Convert sigmoid probabilities into binary classes
    y_pred = (y_pred >= 0.5).astype(int).ravel()
    
    print(classification_report(
        y_true=y_test,
        y_pred=y_pred,
        zero_division=0
    ))    
    
    model.evaluate(X_test,y_test)        
    
    return history

In [19]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-3
)
callbacks = [reduce_lr]
history1 = train_evaluate_model(    
    optimizer=optimizer,
    loss_function="binary_crossentropy",
    X_train=X_train_combined,
    y_train=y_train_combined,    
    X_test=X_test_combined,
    y_test=y_test_combined,
    epochs=10,
    class_weights=class_weight_dict,
    callbacks=callbacks
)

Epoch 1/10


I0000 00:00:1792602489.555547     336 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


337/337 ━━━━━━━━━━━━━━━━━━━━ 40s 76ms/step - accuracy: 0.9882 - loss: 0.0320 - val_accuracy: 0.9971 - val_loss: 0.0078 - learning_rate: 0.0010
Epoch 2/10
337/337 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9979 - loss: 0.0074 - val_accuracy: 0.9966 - val_loss: 0.0083 - learning_rate: 0.0010
Epoch 3/10
337/337 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9985 - loss: 0.0042 - val_accuracy: 0.9966 - val_loss: 0.0101 - learning_rate: 0.0010
Epoch 4/10
333/337 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9992 - loss: 0.0027
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
337/337 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.9994 - loss: 0.0027 - val_accuracy: 0.9971 - val_loss: 0.0072 - learning_rate: 0.0010
Epoch 5/10
337/337 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.9996 - loss: 0.0012 - val_accuracy: 0.9983 - val_loss: 0.0062 - learning_rate: 5.0000e-04
Epoch 6/10
337/337 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9997 - loss: 9.1112e-0

In [20]:
import json

with open("nails_disease_gatekeeper_model_history.json", "w") as f:
    json.dump(history1.history, f)

In [21]:
with open("nails_disease_gatekeeper_model_history.json", "r") as f:
    history_dict = json.load(f)

In [22]:
history_dict

{'accuracy': [0.9882145524024963,
  0.997865617275238,
  0.998515248298645,
  0.9993504285812378,
  0.9996287822723389,
  0.9997215867042542,
  1.0,
  0.9996287822723389,
  0.9999071955680847,
  1.0],
 'loss': [0.03196107596158981,
  0.0074019357562065125,
  0.004194502718746662,
  0.0026905369013547897,
  0.0011864134576171637,
  0.0009111175895668566,
  0.0003913648833986372,
  0.0007348135113716125,
  0.00042562547605484724,
  0.00023743926431052387],
 'val_accuracy': [0.9971412420272827,
  0.9965694546699524,
  0.9965694546699524,
  0.9971412420272827,
  0.9982847571372986,
  0.9977129697799683,
  0.9977129697799683,
  0.9971412420272827,
  0.9977129697799683,
  0.9982847571372986],
 'val_loss': [0.007839110679924488,
  0.008332420140504837,
  0.010142065584659576,
  0.007207857444882393,
  0.006241893395781517,
  0.007151884026825428,
  0.007052288856357336,
  0.009928070940077305,
  0.007196728605777025,
  0.006525352597236633],
 'learning_rate': [0.0010000000474974513,
  0.00100

In [23]:
model.save("nails_disease_gatekeeper_model.keras")

In [15]:
del X_train_nails,X_train_no_nails,X_test_nails,X_test_no_nails,y_test_nails,y_test_no_nails,y_train_nails,y_train_no_nails,dataset
gc.collect()

8